In [2]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [3]:
using JLD2
using QuadGK
using Random
using StaticArrays

In [4]:
data_dir = joinpath(dirname(@__DIR__), "test", "data")
mkpath(data_dir)

l1_file = joinpath(data_dir, "l1_test_data.jld2")
l2_file = joinpath(data_dir, "l2_test_data.jld2")


Random.seed!(18)
tol = eps()
imax = 1e4
qorder = 26


function mod_quadgk(f, a, b; rtol=sqrt(eps()), atol=0, maxevals=10^7, order=7)
    # Put 26 as the first try.
    qorder_vals = [[26, 25, 24]; collect(27:34)]
    
    if !(order in qorder_vals)
        pushfirst!(qorder_vals, order)
    end

    ∫f = err = nothing
    
    for qo in qorder_vals
        ∫f, err, count = quadgk_count(f, a, b; rtol=rtol, atol=atol, maxevals=maxevals, order=qo)
        if count < maxevals
            return ∫f, err
        end
    end

    @warn "Reached the maximum number of function evaluations" #maxlog=1
    
    return ∫f, err
end


function L₁(x::AbstractVector{<:Real})
    A, B, H = x
    
    f(u) = (u + H) / (u - H - (u + H)*exp(-2u))
    f₃(u) = 2 / (u - H - (u + H)*exp(-2u))^2
    f₃₃(u) = 4*(1 + exp(-2u)) / (u - H - (u + H)*exp(-2u))^3
    
    g(u) = exp(-u*(2+B)) + exp(-u*(2-B))
    g₂(u) = exp(-u*(2-B)) - exp(-u*(2+B))
    g₂₂(u) = exp(-u*(2+B)) + exp(-u*(2-B))
    
    h(u) = cos(u*A)
    h₁(u) = -sin(u*A)
    h₁₁(u) = -cos(u*A)
    
    p(u) = (f(u) * g(u) * h(u) + exp(-u)) / u

    p₁(u) = f(u) * g(u) * h₁(u)
    p₂(u) = f(u) * g₂(u) * h(u)
    p₃(u) = f₃(u) * g(u) * h(u)

    p₁₁(u) = f(u) * g(u) * h₁₁(u) * u
    p₁₂(u) = f(u) * g₂(u) * h₁(u) * u
    p₁₃(u) = f₃(u) * g(u) * h₁(u) * u

    p₂₁(u) = f(u) * g₂(u) * h₁(u) * u
    p₂₂(u) = f(u) * g₂₂(u) * h(u) * u
    p₂₃(u) = f₃(u) * g₂(u) * h(u) * u

    p₃₁(u) = f₃(u) * g(u) * h₁(u) * u
    p₃₂(u) = f₃(u) * g₂(u) * h(u) * u
    p₃₃(u) = f₃₃(u) * g(u) * h(u)
    
    path = (0.0, H+im, H+1.0, Inf)

    y = 0.0
    y₁, y₂, y₃ = [0.0 for _ in 1:3]
    y₁₁, y₁₂, y₁₃, y₂₂, y₂₃, y₃₃ = [0.0 for _ in 1:6]

    for i in 1:length(path)-1
        y += mod_quadgk(p, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        
        y₁ += mod_quadgk(p₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂ += mod_quadgk(p₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃ += mod_quadgk(p₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]

        y₁₁ += mod_quadgk(p₁₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₂ += mod_quadgk(p₁₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₃ += mod_quadgk(p₁₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₂ += mod_quadgk(p₂₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₃ += mod_quadgk(p₂₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃₃ += mod_quadgk(p₃₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
    end

    y = real(y)
    
    ∇y = real(SA[y₁, y₂, y₃])
    
    Hy = real(SA[y₁₁ y₁₂ y₁₃;
                 y₁₂ y₂₂ y₂₃;
                 y₁₃ y₂₃ y₃₃])
    
    return y, ∇y, Hy
end


function L₂(x::AbstractVector{<:Real})
    A, B, H = x
    
    # Auxilary variables for f, g and its derivatives
    a(u) = u+H
    b(u) = u-H
    c(u) = (b(u) - a(u)*exp(-2u))
    d(u) = b(u)^2 - a(u)*b(u)*exp(-2u)
    e(u) = b(u)^2 + d(u)
    dH(u) = -2*(b(u) - H*exp(-2u))
    
    f(u)   = a(u) / c(u)
    f₃(u)  = 2 / c(u)^2
    f₃₃(u) = 4*(1 + exp(-2u)) / c(u)^3
    
    g(u)   = a(u)^2 / d(u)
    g₃(u)  = 2*a(u) * (b(u)^2 + d(u)) / (b(u) * d(u)^2)
    g₃₃(u) = 2*(-2a(u)*b(u)*e(u)*dH(u) + a(u)*d(u)*e(u) + b(u)*d(u)*(e(u) - a(u)*(2b(u)-dH(u)))) / (b(u)^2 * d(u)^3)
    
    p(u)   = exp(-u*(2+B))
    p₂(u)  = -exp(-u*(2+B))
    p₂₂(u) = u*exp(-u*(2+B))
    
    q(u)   = exp(-u*(4-B))
    q₂(u)  = exp(-u*(4-B))
    q₂₂(u) = u*exp(-u*(4-B))
    
    r(u)   = cos(u*A)
    r₁(u)  = -sin(u*A)
    r₁₁(u) = -u*cos(u*A)
    
    h(u) = (f(u) * p(u) + g(u) * q(u)) * r(u) / u
    
    h₁(u) = (f(u) * p(u) + g(u) * q(u)) * r₁(u)
    h₂(u) = (f(u) * p₂(u) + g(u) * q₂(u)) * r(u)
    h₃(u) = (f₃(u) * p(u) + g₃(u) * q(u)) * r(u)
    
    h₁₁(u) = (f(u) * p(u) + g(u) * q(u)) * r₁₁(u)
    h₁₂(u) = (f(u) * p₂(u) + g(u) * q₂(u)) * r₁(u) * u
    h₁₃(u) = (f₃(u) * p(u) + g₃(u) * q(u)) * r₁(u) * u
    h₂₂(u) = (f(u) * p₂₂(u) + g(u) * q₂₂(u)) * r(u)
    h₂₃(u) = (f₃(u) * p₂(u) + g₃(u) * q₂(u)) * r(u) * u
    h₃₃(u) = (f₃₃(u) * p(u) + g₃₃(u) * q(u)) * r(u)
    
    path = (0.0, H+im, H+1.0, Inf)

    y = 0.0
    y₁, y₂, y₃ = [0.0 for _ in 1:3]
    y₁₁, y₁₂, y₁₃, y₂₂, y₂₃, y₃₃ = [0.0 for _ in 1:6]

    for i in 1:length(path)-1
        y += mod_quadgk(h, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        
        y₁ += mod_quadgk(h₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂ += mod_quadgk(h₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃ += mod_quadgk(h₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]

        y₁₁ += mod_quadgk(h₁₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₂ += mod_quadgk(h₁₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₃ += mod_quadgk(h₁₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₂ += mod_quadgk(h₂₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₃ += mod_quadgk(h₂₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃₃ += mod_quadgk(h₃₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
    end

    y = real(y)
    
    ∇y = real(SA[y₁, y₂, y₃])
    
    Hy = real(SA[y₁₁ y₁₂ y₁₃;
                 y₁₂ y₂₂ y₂₃;
                 y₁₃ y₂₃ y₃₃])
    
    return y, ∇y, Hy
end;

In [7]:
x = SA[0.1, 0.5, 0.3]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println("quick test\n")
println(y₁)
println(∇y₁)
println(Hy₁)
println()
println(y₂)
println(∇y₂)
println(Hy₂)

quick test

-1.00169204874032
[-0.06799288769514582, 0.28012825734268715, 0.15870001060024233]
[-0.6732390207590058 -0.07372986793058411 0.08936761425251266; -0.07372986793058411 0.6732390207590058 -0.5061078894933684; 0.08936761425251266 -0.5061078894933684 0.11516526881129213]

-1.039841625349199
[0.012907603643945642, -0.2727287078066397, 6.283285000126575]
[0.13029525572290562 0.022550617007363834 0.1454782528537148; 0.022550617007363834 -0.13029525572290554 1.700800437397748; 0.1454782528537148 1.700800437397748 -31.315956117865]


In [8]:
x = SA[0.25, 0.8, 1.1]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println("quick test\n")
println(y₁)
println(∇y₁)
println(Hy₁)
println()
println(y₂)
println(∇y₂)
println(Hy₂)

quick test

-0.8837883138682058
[-0.07409454405793456, -0.17107667328815965, 0.09175971102293126]
[-0.16686752691186219 -0.53617065306508 0.27473963213678465; -0.53617065306508 0.16686752691186219 -0.8138354032975751; 0.27473963213678465 -0.8138354032975751 0.17663352106712116]

0.4398143782891311
[0.0907988382432817, 0.7101391159785377, 0.07700007325499235]
[0.3525174506247106 0.09402034872793776 -0.14479634555981696; 0.09402034872793776 -0.3525174506247106 0.6970968810286419; -0.14479634555981696 0.6970968810286419 -1.5535874769762161]


In [9]:
x = SA[0.3, 0.9, 2.0]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println("quick test\n")
println(y₁)
println(∇y₁)
println(Hy₁)
println()
println(y₂)
println(∇y₂)
println(Hy₂)

quick test

-0.8017542913788627
[0.20771272507576805, -0.8609320364371307, 0.20604384897652497]
[0.9580372469441354 -0.46741888653188723 0.4136727936595598; -0.46741888653188723 -0.9580372469441354 -0.49056795881628307; 0.4136727936595598 -0.49056795881628307 0.15041698412155002]

0.3419667870255596
[-0.02353395200924641, 0.8427205386853558, -0.2656520689035949]
[-0.09423831750202116 -0.07784123478475649 -0.0740595880107929; -0.07784123478475649 0.09423831750202116 -0.15905658393835645; -0.0740595880107929 -0.15905658393835645 0.13077382962819742]


In [8]:
lb = SA[0.0, 0.0, 1e-2]
ub₁ = SA[0.5, 1.0, 7.0]
ub₂ = SA[0.5, 2.0, 7.0]

npoints = 1000

x1 = [lb .+ rand(SVector{3, Float64}) .* (ub₁ .- lb) for _ in 1:npoints]
x2 = [lb .+ rand(SVector{3, Float64}) .* (ub₂ .- lb) for _ in 1:npoints];

In [9]:
L1 = Vector{Float64}(undef, npoints)
∇L1 = Matrix{Float64}(undef, npoints, 3)
HL1 = Array{Float64, 3}(undef, npoints, 3, 3)

L2 = Vector{Float64}(undef, npoints)
∇L2 = Matrix{Float64}(undef, npoints, 3)
HL2 = Array{Float64, 3}(undef, npoints, 3, 3);

In [10]:
for i in 1:npoints
    y₁, ∇y₁, Hy₁ = L₁(x1[i])
    y₂, ∇y₂, Hy₂ = L₂(x2[i])

    L1[i] .= y₁
    ∇L1[i, :] .= ∇y₁
    HL1[i, :, :] .= Hy₁
    
    L2[i] .= y₂
    ∇L2[i, :] .= ∇y₂
    HL2[i, :, :] .= Hy₂

    if mod(i, floor(Int, npoints/10)) == 0
        println("$i of $npoints points computed")
    elseif i == npoints
        println("$i of $npoints points computed")
    end
end

@save l1_file x1 L1 ∇L1 HL1
@save l2_file x2 L2 ∇L2 HL2;

100 of 1000 points computed
200 of 1000 points computed
300 of 1000 points computed
400 of 1000 points computed
500 of 1000 points computed
600 of 1000 points computed
700 of 1000 points computed
800 of 1000 points computed
900 of 1000 points computed
1000 of 1000 points computed
